# CVS-Act synthetic-GT held-out-test metrics excluding dev videos

Recomputes the small-scale Baseline vs SurGent synthetic-GT table and the larger pooled baseline table after excluding the 10-video dev split from the human-GT 30-video set. It also writes split-level synthetic-GT clip counts to CSV and TeX.

In [1]:

from pathlib import Path
from collections import defaultdict
import csv
import hashlib
import json
import random
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT_DIR = Path('/mnt/md0/weiqiuy/surgent')
OUT_DIR = ROOT_DIR / 'notebooks/artifacts/cvs_act_synthetic_gt_test_excluding_dev_stats'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [ROOT_DIR / 'scripts/eval', ROOT_DIR / 'src']:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from evaluate_cvs_act_v1_combined import load_synthetic_action_records, collect_prediction_groups
from evaluate_audit_v11_simple_recommendation import gt_records_by_video
from evaluate_cvs_act_v1_onset_pointwise import (
    collect_eval_points,
    evaluate_points,
    resolve_gt_rows,
    present_label_rows,
    label_value,
    label_text,
)

SPLIT_SEED = 20260610
SPLIT_CSV = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_current_best_prompt_summary/validation_test_video_split_seed20260610.csv'
SYNTHETIC_TEST_JSONL = ROOT_DIR / 'hf_repos/cvs-act/sages_synthetic/test.jsonl'

split_df = pd.read_csv(SPLIT_CSV)
dev_video_ids = set(split_df.loc[split_df['split'].eq('validation'), 'video_id'].astype(str))
test_video_ids = set(split_df.loc[split_df['split'].eq('test'), 'video_id'].astype(str))
human_gt_video_ids_30 = dev_video_ids | test_video_ids
assert len(dev_video_ids) == 10
assert len(test_video_ids) == 20
assert len(human_gt_video_ids_30) == 30

synthetic_records_all = load_synthetic_action_records(SYNTHETIC_TEST_JSONL)
for record in synthetic_records_all:
    record.setdefault('clip_level', 'coarse')

synthetic_records_dev10 = [r for r in synthetic_records_all if str(r.get('video_id')) in dev_video_ids]
synthetic_records_test20 = [r for r in synthetic_records_all if str(r.get('video_id')) in test_video_ids]
synthetic_records_human30 = [r for r in synthetic_records_all if str(r.get('video_id')) in human_gt_video_ids_30]

synthetic_model_filter = {'claude-haiku-4-5-20251001', 'gemini-2.5-flash', 'gpt-5.4-mini'}
synthetic_taxonomy_filter = {'cvs_act_current_simple_v1'}

variant_display = {
    'cot_fixedk3_norecdescs_fmeta': 'Baseline',
    'pref-cvs_arec_steps5_fixedk3_norecdescs_fmeta': 'SurGent',
    'cot_fixedk3_no_cvs_norecdescs_fmeta': 'Baseline no-CVS',
    'cot_fixedk3_no_cvs_no_desc_norecdescs_fmeta': 'Baseline no-CVS no-desc',
    'cot_fixedk3_no_cvs_no_guideline_norecdescs_fmeta': 'Baseline no-CVS no-guideline',
    'cot_fixedk3_norecdescs_fmeta_arecrules-conservative-visible': 'Baseline + action rules',
}
model_display = {
    'claude-haiku-4-5-20251001': 'Claude-Haiku-4.5',
    'gemini-2.5-flash': 'Gemini-2.5-flash',
    'gpt-5.4-mini': 'GPT-5.4-mini',
}
model_order_full = ['Claude-Haiku-4.5', 'Gemini-2.5-flash', 'GPT-5.4-mini']
final_components = [('left', 'exact'), ('right', 'medium'), ('camera', 'coarse')]
BOOTSTRAP_N = 1000

baseline_surgent_dirs = [
    ROOT_DIR / 'outputs/cot_audit_v11_simple/cot_fixedk3_norecdescs_fmeta',
    ROOT_DIR / 'outputs/surgent_sequential_agent_audit_v11_simple/cot/pref-cvs_arec_steps5_fixedk3_norecdescs_fmeta',
]
baseline_only_dirs = [
    ROOT_DIR / 'outputs/cot_audit_v11_simple/cot_fixedk3_norecdescs_fmeta',
    ROOT_DIR / 'outputs/cot_audit_v11_simple/cot_fixedk3_no_cvs_norecdescs_fmeta',
    ROOT_DIR / 'outputs/cot_audit_v11_simple/cot_fixedk3_no_cvs_no_desc_norecdescs_fmeta',
    ROOT_DIR / 'outputs/cot_audit_v11_simple/cot_fixedk3_no_cvs_no_guideline_norecdescs_fmeta',
    ROOT_DIR / 'outputs/cot_audit_v11_simple/cot_fixedk3_norecdescs_fmeta_arecrules-conservative-visible',
]


def variant_from_method(method):
    return str(method).split('/')[0]


def model_from_method(method):
    parts = str(method).split('/')
    return parts[1] if len(parts) > 1 else ''


def confusion_count_rows(y_true, y_pred):
    labels = sorted({label_text(v) for v in y_true} | {label_text(v) for v in y_pred})
    rows = []
    for label in labels:
        tp = sum(label_text(t) == label and label_text(p) == label for t, p in zip(y_true, y_pred))
        fp = sum(label_text(t) != label and label_text(p) == label for t, p in zip(y_true, y_pred))
        fn = sum(label_text(t) == label and label_text(p) != label for t, p in zip(y_true, y_pred))
        rows.append({'label': label, 'tp': int(tp), 'fp': int(fp), 'fn': int(fn)})
    return rows


def score_eval_method_video_counts(method_rows):
    counts = []
    for (video_id, actor, granularity), sub_df in method_rows.groupby(['video_id', 'actor', 'granularity'], dropna=False):
        present = sub_df[sub_df['n_present_gt_rows'].fillna(0).astype(int) > 0].copy()
        if present.empty:
            continue
        y_true = []
        y_pred = []
        for row in present.itertuples(index=False):
            gt_rows = resolve_gt_rows(row.point, actor, int(row.true_onset_frame)) if hasattr(row, 'point') else row.gt_rows
            # In this notebook rows are dicts converted to DataFrame; use stored lists directly when available.
        
    return counts


def score_eval_method_rows(method_rows):
    rows = []
    df = pd.DataFrame(method_rows)
    if df.empty:
        return rows
    for actor, granularity in final_components:
        sub = df[(df['clip_level'] == 'coarse') & (df['point_source'] == 'coarse_actor_onset') & (df['actor'] == actor)].copy()
        if sub.empty:
            continue
        y_true = []
        y_pred = []
        for item in sub.to_dict('records'):
            gt_rows = item['gt_rows']
            present_rows = present_label_rows(gt_rows, actor)
            if not present_rows:
                continue
            pred_label = item[f'pred_label_{granularity}']
            gt_labels = [label_value(row, actor, granularity) for row in present_rows]
            # Resolve to a matching GT label if several rows are active; otherwise deterministic first label.
            matched = None
            for gt_label in gt_labels:
                if label_text(pred_label) == label_text(gt_label):
                    matched = gt_label
                    break
            if matched is None:
                matched = sorted(gt_labels, key=label_text)[0]
            y_true.append(matched)
            y_pred.append(pred_label)
        rows.append({
            'component': f'{actor}_{granularity}',
            'actor': actor,
            'granularity': granularity,
            'counts': confusion_count_rows(y_true, y_pred),
            'n': len(y_true),
        })
    return rows


def component_macro_f1_from_counts(counts):
    vals = []
    for row in counts:
        tp = row['tp']; fp = row['fp']; fn = row['fn']
        denom = 2 * tp + fp + fn
        vals.append((2 * tp / denom) if denom else 0.0)
    return float(np.mean(vals)) if vals else np.nan


def final_score_from_count_table(count_table):
    vals = [component_macro_f1_from_counts(row['counts']) for row in count_table]
    vals = [v for v in vals if np.isfinite(v)]
    return float(np.mean(vals)) if vals else np.nan


def build_video_count_arrays(method_rows):
    df = pd.DataFrame(method_rows)
    if df.empty:
        return []
    out = []
    for video_id, video_df in df.groupby('video_id'):
        out.append({'video_id': video_id, 'count_table': score_eval_method_rows(video_df.to_dict('records'))})
    return out


def macro_f1_from_summed_counts(tp_sum, fp_sum, fn_sum):
    vals = []
    for label in sorted(set(tp_sum) | set(fp_sum) | set(fn_sum)):
        tp = tp_sum.get(label, 0); fp = fp_sum.get(label, 0); fn = fn_sum.get(label, 0)
        denom = 2 * tp + fp + fn
        vals.append((2 * tp / denom) if denom else 0.0)
    return float(np.mean(vals)) if vals else np.nan


def bootstrap_eval_method_final_score_se(method_rows, seed, n_boot=BOOTSTRAP_N):
    video_tables = build_video_count_arrays(method_rows)
    if len(video_tables) < 2:
        return np.nan
    rng = random.Random(seed)
    scores = []
    for _ in range(n_boot):
        sampled = [rng.choice(video_tables)['count_table'] for _ in range(len(video_tables))]
        component_scores = []
        for component in [f'{actor}_{granularity}' for actor, granularity in final_components]:
            tp_sum = defaultdict(int); fp_sum = defaultdict(int); fn_sum = defaultdict(int)
            for table in sampled:
                row = next((r for r in table if r['component'] == component), None)
                if row is None:
                    continue
                for count in row['counts']:
                    label = count['label']
                    tp_sum[label] += count['tp']; fp_sum[label] += count['fp']; fn_sum[label] += count['fn']
            component_scores.append(macro_f1_from_summed_counts(tp_sum, fp_sum, fn_sum))
        vals = [v for v in component_scores if np.isfinite(v)]
        if vals:
            scores.append(float(np.mean(vals)))
    return float(np.std(scores, ddof=1)) if len(scores) > 1 else np.nan


def score_eval_rows(eval_rows, *, title):
    rows = []
    df = pd.DataFrame(eval_rows)
    if df.empty:
        return pd.DataFrame(rows)
    for method, method_df in df.groupby('method'):
        count_table = score_eval_method_rows(method_df.to_dict('records'))
        component_scores = {row['component']: component_macro_f1_from_counts(row['counts']) for row in count_table}
        component_ns = {row['component']: row['n'] for row in count_table}
        variant = variant_from_method(method)
        model = model_from_method(method)
        rows.append({
            'analysis': title,
            'gt_source': 'v3.1 structured LLM extract + tool fix synthetic GT',
            'method': method,
            'variant': variant,
            'variant_label': variant_display.get(variant, variant),
            'model': model,
            'model_name': model_display.get(model, model),
            'system_label': f'{model_display.get(model, model)} / {variant_display.get(variant, variant)}',
            'n_videos': int(method_df['video_id'].nunique()),
            'left_exact': component_scores.get('left_exact', np.nan),
            'right_medium': component_scores.get('right_medium', np.nan),
            'camera_coarse': component_scores.get('camera_coarse', np.nan),
            'left_exact_n': component_ns.get('left_exact', 0),
            'right_medium_n': component_ns.get('right_medium', 0),
            'camera_coarse_n': component_ns.get('camera_coarse', 0),
            'final_score': final_score_from_count_table(count_table),
            'final_score_se': bootstrap_eval_method_final_score_se(method_df.to_dict('records'), seed=SPLIT_SEED + int(hashlib.sha256(str(method).encode('utf-8')).hexdigest()[:8], 16) % 100000),
            'n_components': len(count_table),
        })
    return pd.DataFrame(rows)


def evaluate_synthetic_records(records, output_dirs):
    gt_by_video = gt_records_by_video(records)
    frames_by_method, paths_by_method, skipped = collect_prediction_groups(
        output_dirs,
        gt_by_video,
        include_partial=False,
        modified_since=None,
        model_filter=synthetic_model_filter,
        taxonomy_filter=synthetic_taxonomy_filter,
        verbose=False,
    )
    eval_points = collect_eval_points(records)
    rows = evaluate_points(frames_by_method, eval_points)
    # Keep the resolved rows directly so scoring can handle multi-active GT rows.
    for row in rows:
        row['point'] = next(
            p for p in eval_points
            if str(p['record']['video_id']) == str(row['video_id'])
            and p['record'].get('example_id') == row.get('example_id')
            and p['actor'] == row['actor']
            and p['point_source'] == row['point_source']
            and int(p['point_frame']) == int(row['true_onset_frame'])
        )
        row['gt_rows'] = resolve_gt_rows(row['point'], row['actor'], int(row['true_onset_frame']))
    inventory = []
    for method, by_video in sorted(frames_by_method.items()):
        variant = variant_from_method(method)
        model = model_from_method(method)
        inventory.append({
            'method': method,
            'variant': variant,
            'variant_label': variant_display.get(variant, variant),
            'model': model,
            'model_name': model_display.get(model, model),
            'n_prediction_videos': len(by_video),
        })
    return rows, pd.DataFrame(inventory), skipped


def majority_component_scores_from_synthetic_records(records, analysis_label):
    eval_points = collect_eval_points(records)
    rows = []
    values = []
    for actor, granularity in final_components:
        y_true = []
        gt_texts = []
        for point in eval_points:
            if point['record'].get('clip_level', 'coarse') != 'coarse':
                continue
            if point['point_source'] != 'coarse_actor_onset' or point['actor'] != actor:
                continue
            gt_rows = resolve_gt_rows(point, actor, int(point['point_frame']))
            present_rows = present_label_rows(gt_rows, actor)
            if not present_rows:
                continue
            labels = [label_value(row, actor, granularity) for row in present_rows]
            chosen = sorted(labels, key=label_text)[0]
            y_true.append(chosen)
            gt_texts.append(label_text(chosen))
        majority_text = pd.Series(gt_texts).mode().iat[0] if gt_texts else ''
        y_pred = [majority_text] * len(gt_texts)
        counts = confusion_count_rows([label_text(v) for v in y_true], y_pred)
        score = component_macro_f1_from_counts(counts)
        rows.append({
            'analysis': analysis_label,
            'gt_source': 'v3.1 structured LLM extract + tool fix synthetic GT',
            'component': f'{actor}_{granularity}',
            'actor': actor,
            'granularity': granularity,
            'majority_label': majority_text,
            'majority_label_macro_f1': score,
            'n': len(y_true),
            'n_videos': len({str(r.get('video_id')) for r in records}),
            'synthetic_gt_records_n': len(records),
        })
        values.append(score)
    rows.append({
        'analysis': analysis_label,
        'gt_source': 'v3.1 structured LLM extract + tool fix synthetic GT',
        'component': 'final_score',
        'actor': 'aggregate',
        'granularity': 'final',
        'majority_label': '',
        'majority_label_macro_f1': float(np.nanmean(values)),
        'n': int(sum(row['n'] for row in rows)),
        'n_videos': len({str(r.get('video_id')) for r in records}),
        'synthetic_gt_records_n': len(records),
    })
    return pd.DataFrame(rows)


def final_majority_score(majority_df):
    row = majority_df[majority_df['component'].eq('final_score')]
    return float(row['majority_label_macro_f1'].iloc[0]) if not row.empty else np.nan


def prefix_eval_video_ids(rows, source_label):
    out = []
    for row in rows:
        copied = dict(row)
        copied['original_video_id'] = str(row.get('video_id'))
        copied['video_id'] = f'{source_label}::{row.get("video_id")}'
        copied['pooled_source'] = source_label
        out.append(copied)
    return out

# Small scale: exclude the 10 dev videos, so this is the held-out 20-video human-GT test split.
bs_test_eval_rows, bs_test_inventory, bs_test_skipped = evaluate_synthetic_records(synthetic_records_test20, baseline_surgent_dirs)
bs_test_score_df = score_eval_rows(bs_test_eval_rows, title='baseline_vs_surgent_test20_excluding_dev10')
bs_test_score_df['requested_video_set_n'] = len(test_video_ids)
bs_test_score_df['synthetic_gt_records_n'] = len(synthetic_records_test20)
bs_test_majority_df = majority_component_scores_from_synthetic_records(synthetic_records_test20, 'baseline_vs_surgent_test20_excluding_dev10')

# Larger baseline: recover the old plain-baseline larger coverage, then remove dev10.
# The larger65 run already covers all 30 human-GT videos, including the held-out test20,
# so do not append test20 again here.
bo_eval_rows, bo_inventory, bo_skipped = evaluate_synthetic_records(synthetic_records_all, baseline_only_dirs)
plain_baseline_variant = 'cot_fixedk3_norecdescs_fmeta'
bo_plain_baseline_eval_rows_all65 = [row for row in bo_eval_rows if variant_from_method(row.get('method')) == plain_baseline_variant]
bo_plain_baseline_eval_rows = [
    row for row in bo_plain_baseline_eval_rows_all65
    if str(row.get('video_id')) not in dev_video_ids
]
larger65_all_video_ids = sorted({str(row['video_id']) for row in bo_plain_baseline_eval_rows_all65})
larger65_excluding_dev_video_ids = sorted({str(row['video_id']) for row in bo_plain_baseline_eval_rows})
larger65_all_records = [r for r in synthetic_records_all if str(r.get('video_id')) in set(larger65_all_video_ids)]
larger65_excluding_dev_records = [r for r in synthetic_records_all if str(r.get('video_id')) in set(larger65_excluding_dev_video_ids)]
outside_human30_video_ids = sorted(set(larger65_all_video_ids) - human_gt_video_ids_30)
outside_human30_records = [r for r in synthetic_records_all if str(r.get('video_id')) in set(outside_human30_video_ids)]

bo_test_combined_eval_rows = prefix_eval_video_ids(bo_plain_baseline_eval_rows, 'larger65_minus_dev10')
bo_test_combined_inventory = pd.DataFrame([
    {'pooled_source': 'larger65_minus_dev10', 'n_eval_rows': len(bo_plain_baseline_eval_rows), 'n_videos': len(larger65_excluding_dev_video_ids)},
])
bo_test_combined_score_df = score_eval_rows(bo_test_combined_eval_rows, title='baseline_larger65_minus_dev10')
bo_test_combined_score_df['pooled_larger_excluding_dev_n_videos'] = int(bo_test_combined_inventory.loc[bo_test_combined_inventory['pooled_source'].eq('larger65_minus_dev10'), 'n_videos'].iloc[0])
bo_test_combined_score_df['pooled_total_n_videos'] = bo_test_combined_score_df['pooled_larger_excluding_dev_n_videos']
bo_test_combined_majority_df = majority_component_scores_from_synthetic_records(larger65_excluding_dev_records, 'baseline_larger65_minus_dev10')

# Clip stats for requested splits.
def clip_stats(records, split_name, expected_video_ids):
    counts = pd.Series([str(r.get('video_id')) for r in records]).value_counts() if records else pd.Series(dtype=int)
    return {
        'split': split_name,
        'n_videos': len(expected_video_ids),
        'n_videos_with_gt_clips': int(counts.shape[0]),
        'n_clips': int(len(records)),
        'clips_per_video_mean': float(len(records) / len(expected_video_ids)) if expected_video_ids else np.nan,
        'clips_per_video_mean_nonempty': float(counts.mean()) if len(counts) else np.nan,
        'clips_per_video_min_nonempty': int(counts.min()) if len(counts) else 0,
        'clips_per_video_max_nonempty': int(counts.max()) if len(counts) else 0,
    }

clip_stats_df = pd.DataFrame([
    clip_stats(synthetic_records_dev10, 'dev10_human_gt_split', dev_video_ids),
    clip_stats(synthetic_records_test20, 'test20_human_gt_split', test_video_ids),
    clip_stats(larger65_all_records, 'larger65_plain_baseline_coverage_all', set(larger65_all_video_ids)),
    clip_stats(larger65_excluding_dev_records, 'larger65_plain_baseline_coverage_excluding_dev10', set(larger65_excluding_dev_video_ids)),
    clip_stats(outside_human30_records, 'outside_human30_subset_within_larger65', set(outside_human30_video_ids)),
])

# Save CSVs.
paths = {}
paths['small_scores_csv'] = OUT_DIR / 'synthetic_gt_baseline_vs_surgent_test20_excluding_dev10_final_scores.csv'
paths['small_inventory_csv'] = OUT_DIR / 'synthetic_gt_baseline_vs_surgent_test20_excluding_dev10_inventory.csv'
paths['pooled_scores_csv'] = OUT_DIR / 'synthetic_gt_baseline_only_larger_plus_test20_excluding_dev10_final_scores.csv'
paths['pooled_inventory_csv'] = OUT_DIR / 'synthetic_gt_baseline_only_larger_plus_test20_excluding_dev10_inventory.csv'
paths['majority_csv'] = OUT_DIR / 'synthetic_gt_majority_baseline_excluding_dev10_final_scores.csv'
paths['clip_stats_csv'] = OUT_DIR / 'synthetic_gt_split_clip_counts_excluding_dev10.csv'
paths['small_scores_tex'] = OUT_DIR / 'synthetic_gt_baseline_vs_surgent_test20_excluding_dev10_final_scores.tex'
paths['pooled_scores_tex'] = OUT_DIR / 'synthetic_gt_baseline_only_larger_plus_test20_excluding_dev10_final_scores.tex'
paths['clip_stats_tex'] = OUT_DIR / 'synthetic_gt_split_clip_counts_excluding_dev10.tex'
paths['small_scores_png'] = OUT_DIR / 'synthetic_gt_baseline_vs_surgent_test20_excluding_dev10_final_scores.png'
paths['small_scores_pdf'] = OUT_DIR / 'synthetic_gt_baseline_vs_surgent_test20_excluding_dev10_final_scores.pdf'
paths['pooled_scores_png'] = OUT_DIR / 'synthetic_gt_baseline_only_larger_plus_test20_excluding_dev10_final_scores.png'
paths['pooled_scores_pdf'] = OUT_DIR / 'synthetic_gt_baseline_only_larger_plus_test20_excluding_dev10_final_scores.pdf'

bs_test_score_df.to_csv(paths['small_scores_csv'], index=False)
bs_test_inventory.to_csv(paths['small_inventory_csv'], index=False)
bo_test_combined_score_df.to_csv(paths['pooled_scores_csv'], index=False)
bo_test_combined_inventory.to_csv(paths['pooled_inventory_csv'], index=False)
pd.concat([bs_test_majority_df, bo_test_combined_majority_df], ignore_index=True).to_csv(paths['majority_csv'], index=False)
clip_stats_df.to_csv(paths['clip_stats_csv'], index=False)

small_tex_df = bs_test_score_df[['model_name', 'variant_label', 'final_score', 'final_score_se', 'n_videos', 'left_exact', 'right_medium', 'camera_coarse']].sort_values(['model_name', 'variant_label']).copy()
pooled_tex_df = bo_test_combined_score_df[['model_name', 'variant_label', 'final_score', 'final_score_se', 'n_videos', 'pooled_larger_excluding_dev_n_videos', 'pooled_total_n_videos', 'left_exact', 'right_medium', 'camera_coarse']].sort_values('final_score', ascending=False).copy()
clip_label_map = {
    'dev10_human_gt_split': 'Dev (human-GT)',
    'test20_human_gt_split': 'Held-out test (human-GT)',
    'larger65_plain_baseline_coverage_all': 'Larger synthetic run',
    'larger65_plain_baseline_coverage_excluding_dev10': 'Larger synthetic run, no dev',
    'outside_human30_subset_within_larger65': 'Additional synthetic-only',
}
clip_tex_df = clip_stats_df.assign(Split=clip_stats_df['split'].map(clip_label_map).fillna(clip_stats_df['split']))[[
    'Split',
    'n_videos',
    'n_clips',
    'clips_per_video_mean',
    'clips_per_video_min_nonempty',
    'clips_per_video_max_nonempty',
]].rename(columns={
    'n_videos': 'Videos',
    'n_clips': 'Clips',
    'clips_per_video_mean': 'Clips/video',
    'clips_per_video_min_nonempty': 'Min',
    'clips_per_video_max_nonempty': 'Max',
})

tex_float_fmt = '%.3f'
paths['small_scores_tex'].write_text(small_tex_df.to_latex(index=False, float_format=tex_float_fmt, caption='Synthetic-GT final scores on held-out human-GT test videos, excluding the 10-video dev split.', label='tab:cvs-act-synthetic-test20-excluding-dev'))
paths['pooled_scores_tex'].write_text(pooled_tex_df.to_latex(index=False, float_format=tex_float_fmt, caption='Pooled synthetic-GT plain-baseline scores using the 65-video larger run after excluding the 10-video dev split.', label='tab:cvs-act-synthetic-pooled-larger65-excluding-dev'))
paths['clip_stats_tex'].write_text(clip_tex_df.to_latex(index=False, float_format=tex_float_fmt, escape=True, caption='Synthetic-GT clip counts by split after separating the 10-video dev split from the held-out test split.', label='tab:cvs-act-synthetic-split-clip-counts'))

# Corrected bar charts matching the score tables.
def plot_small_scores(score_df, majority_df, png_path, pdf_path):
    fig, ax = plt.subplots(figsize=(12.8, 7.2))
    plot_df = score_df.copy()
    plot_df['model_name'] = pd.Categorical(plot_df['model_name'], categories=model_order_full, ordered=True)
    plot_df = plot_df.sort_values(['model_name', 'variant_label'])
    y_pos = np.arange(len(model_order_full))
    bar_width = 0.34
    for offset, variant_label, color in [(-bar_width / 2, 'Baseline', '#4C78A8'), (bar_width / 2, 'SurGent', '#F28E2B')]:
        vals, ses, ns = [], [], []
        for model_name in model_order_full:
            row = plot_df[(plot_df['model_name'].astype(str) == model_name) & (plot_df['variant_label'] == variant_label)]
            vals.append(float(row['final_score'].iloc[0]) if not row.empty else np.nan)
            ses.append(float(row['final_score_se'].iloc[0]) if not row.empty and pd.notna(row['final_score_se'].iloc[0]) else 0.0)
            ns.append(int(row['n_videos'].iloc[0]) if not row.empty else 0)
        bars = ax.barh(y_pos + offset, vals, height=bar_width, color=color, label=variant_label, edgecolor='white', linewidth=0.8, xerr=ses, capsize=3)
        for rect, val, se, n_videos in zip(bars, vals, ses, ns):
            if np.isfinite(val):
                ax.text(val + se + 0.006, rect.get_y() + rect.get_height() / 2, f'{val:.3f}  n={n_videos}', ha='left', va='center', fontsize=13)
    majority_score = final_majority_score(majority_df)
    if np.isfinite(majority_score):
        ax.axvline(majority_score, color='#7A5195', linestyle='--', linewidth=1.5, label=f'Majority baseline ({majority_score:.3f})')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(model_order_full, fontsize=15)
    ax.set_xlabel('Final score vs synthetic GT', fontsize=17)
    ax.set_title('Synthetic GT final score on held-out 20 human-GT test videos', fontsize=20, pad=42)
    xmax = max(float(plot_df['final_score'].max()), majority_score if np.isfinite(majority_score) else 0)
    ax.set_xlim(0, max(0.32, xmax + 0.08))
    ax.grid(axis='x', alpha=0.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, -0.24), ncols=3, fontsize=15)
    plt.tight_layout()
    fig.savefig(png_path, dpi=220, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    plt.close(fig)


def plot_pooled_scores(score_df, majority_df, png_path, pdf_path):
    fig, ax = plt.subplots(figsize=(9.6, 4.6))
    plot_df = score_df.sort_values('final_score', ascending=True).copy()
    y_pos = np.arange(len(plot_df))
    colors = plot_df['model_name'].map({'Claude-Haiku-4.5': '#4C78A8', 'Gemini-2.5-flash': '#59A14F', 'GPT-5.4-mini': '#F28E2B'}).fillna('#777777')
    ax.barh(y_pos, plot_df['final_score'], color=colors, edgecolor='white', linewidth=0.8, xerr=plot_df['final_score_se'].fillna(0.0), capsize=3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f'{row.model_name} / {row.variant_label}' for row in plot_df.itertuples()], fontsize=15)
    majority_score = final_majority_score(majority_df)
    if np.isfinite(majority_score):
        ax.axvline(majority_score, color='#7A5195', linestyle='--', linewidth=1.5, label=f'Majority baseline ({majority_score:.3f})')
    ax.set_xlabel('Final score vs synthetic GT', fontsize=17)
    ax.set_title('Baseline pooled synthetic-GT run (larger65 minus dev10)', fontsize=20, pad=42)
    xmax = max(float(plot_df['final_score'].max()), majority_score if np.isfinite(majority_score) else 0)
    ax.set_xlim(0, max(0.32, xmax + 0.07))
    for idx, row in enumerate(plot_df.itertuples()):
        se = row.final_score_se if pd.notna(row.final_score_se) else 0.0
        ax.text(row.final_score + se + 0.004, idx, f'{row.final_score:.3f}  n={row.n_videos}', va='center', fontsize=13)
    ax.grid(axis='x', alpha=0.25)
    if np.isfinite(majority_score):
        ax.legend(frameon=False, loc='lower right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    fig.savefig(png_path, dpi=220, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    plt.close(fig)

plot_small_scores(bs_test_score_df, bs_test_majority_df, paths['small_scores_png'], paths['small_scores_pdf'])
plot_pooled_scores(bo_test_combined_score_df, bo_test_combined_majority_df, paths['pooled_scores_png'], paths['pooled_scores_pdf'])

# Clearer aliases for the corrected de-duplicated larger-minus-dev pool. The legacy
# filenames are retained because earlier SCP commands used them.
alias_paths = {
    'pooled_scores_csv_alias': OUT_DIR / 'synthetic_gt_baseline_only_larger_excluding_dev10_final_scores.csv',
    'pooled_inventory_csv_alias': OUT_DIR / 'synthetic_gt_baseline_only_larger_excluding_dev10_inventory.csv',
    'pooled_scores_tex_alias': OUT_DIR / 'synthetic_gt_baseline_only_larger_excluding_dev10_final_scores.tex',
    'pooled_scores_png_alias': OUT_DIR / 'synthetic_gt_baseline_only_larger_excluding_dev10_final_scores.png',
    'pooled_scores_pdf_alias': OUT_DIR / 'synthetic_gt_baseline_only_larger_excluding_dev10_final_scores.pdf',
}
alias_sources = {
    'pooled_scores_csv_alias': paths['pooled_scores_csv'],
    'pooled_inventory_csv_alias': paths['pooled_inventory_csv'],
    'pooled_scores_tex_alias': paths['pooled_scores_tex'],
    'pooled_scores_png_alias': paths['pooled_scores_png'],
    'pooled_scores_pdf_alias': paths['pooled_scores_pdf'],
}
for key, alias_path in alias_paths.items():
    alias_path.write_bytes(alias_sources[key].read_bytes())
paths.update(alias_paths)

# Summary markdown for quick review.
summary_lines = [
    '# CVS-Act synthetic-GT scores excluding dev10',
    '',
    f'- Split seed: `{SPLIT_SEED}`',
    f'- Dev videos excluded from score tables: `{len(dev_video_ids)}`',
    f'- Held-out human-GT test videos requested: `{len(test_video_ids)}`',
    f'- Held-out human-GT test videos scored in small table: `{int(bs_test_score_df["n_videos"].min())}`-`{int(bs_test_score_df["n_videos"].max())}`',
    f'- Larger plain-baseline videos before dev exclusion: `{len(larger65_all_video_ids)}`',
    f'- Larger plain-baseline videos after dev exclusion: `{len(larger65_excluding_dev_video_ids)}`',
    f'- Pooled larger-minus-dev unique videos scored: `{int(bo_test_combined_score_df["pooled_total_n_videos"].iloc[0])}`',
    '',
    '## Outputs',
]
for key, value in paths.items():
    summary_lines.append(f'- `{key}`: `{value}`')
summary_lines.extend([
    '',
    '## Naming note',
    '',
    'The `larger_plus_test20` pooled filenames are retained for compatibility with earlier SCP commands, but their contents now use the corrected de-duplicated pool: the 65-video larger run after removing the 10 development videos (`N=55`). The held-out 20 videos are not appended a second time. Equivalent clearer aliases are also available with `larger_excluding_dev10` in the filename.',
])
summary_path = OUT_DIR / 'README.md'
summary_path.write_text('\n'.join(summary_lines) + '\n')
paths['summary_md'] = summary_path

print('Small-scale scores excluding dev10:')
print(small_tex_df.to_string(index=False))
print('\nPooled larger65 scores excluding dev10:')
print(pooled_tex_df.to_string(index=False))
print('\nClip stats:')
print(clip_stats_df.to_string(index=False))
print('\nSaved paths:')
for key, value in paths.items():
    print(f'{key}: {value}')


/mnt/md0/weiqiuy/surgent/.venv/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


Small-scale scores excluding dev10:
      model_name variant_label  final_score  final_score_se  n_videos  left_exact  right_medium  camera_coarse
Claude-Haiku-4.5      Baseline     0.226816        0.099281        20    0.180142      0.021138       0.479167
Claude-Haiku-4.5       SurGent     0.192837        0.033131        20    0.182456      0.037081       0.358974
    GPT-5.4-mini      Baseline     0.165405        0.021574        20    0.058687      0.032766       0.404762
    GPT-5.4-mini       SurGent     0.190727        0.025129        20    0.087897      0.039839       0.444444
Gemini-2.5-flash      Baseline     0.144841        0.024031        20    0.073650      0.036547       0.324324
Gemini-2.5-flash       SurGent     0.168877        0.026991        20    0.122394      0.042131       0.342105

Pooled larger65 scores excluding dev10:
      model_name variant_label  final_score  final_score_se  n_videos  pooled_larger_excluding_dev_n_videos  pooled_total_n_videos  left_exact  ri